# ICML 2013 Black Box Learning Challenge - Vision Transformer Solution

## Challenge Overview
- **Goal**: Train a classifier on non-human-readable data without knowing what it represents
- **Data**: ~130K unsupervised examples + labeled training data
- **Metric**: Classification accuracy on private test set
- **Approach**: Vision Transformer (ViT) with self-supervised pre-training

## Strategy
1. Analyze the unknown data to understand its properties
2. Pre-train using Masked Autoencoder (MAE) on 130K unsupervised data
3. Fine-tune Vision Transformer classifier on labeled data
4. Use data augmentation and ensembling for better generalization

## 1. Setup and Imports

In [ ]:
# Install required packages if needed
# !pip install torch torchvision pandas numpy scikit-learn matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 2. Load and Analyze Data

In [ ]:
# Load training data
train_df = pd.read_csv('train.csv')

# Separate features and labels
# Assuming last column is the label
X_train = train_df.iloc[:, :-1].values.astype(np.float32)
y_train = train_df.iloc[:, -1].values.astype(np.int64)

print("Training Data:")
print(f"  Shape: {X_train.shape}")
print(f"  Samples: {X_train.shape[0]}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Classes: {len(np.unique(y_train))}")
print(f"  Class distribution: {np.bincount(y_train)}")

In [ ]:
# Load test data
test_df = pd.read_csv('test.csv')
X_test = test_df.values.astype(np.float32)

print("\nTest Data:")
print(f"  Shape: {X_test.shape}")

In [ ]:
# Load unsupervised data (if available)
try:
    unsup_df = pd.read_csv('unsupervised.csv')
    X_unsup = unsup_df.values.astype(np.float32)
    print("\nUnsupervised Data:")
    print(f"  Shape: {X_unsup.shape}")
    USE_PRETRAIN = True
except FileNotFoundError:
    print("\nUnsupervised data not found. Skipping pre-training.")
    X_unsup = None
    USE_PRETRAIN = False

In [ ]:
# Analyze the data
print("\n" + "="*70)
print("DATA ANALYSIS")
print("="*70)

print(f"\nValue Statistics:")
print(f"  Min: {X_train.min():.6f}")
print(f"  Max: {X_train.max():.6f}")
print(f"  Mean: {X_train.mean():.6f}")
print(f"  Std: {X_train.std():.6f}")

print(f"\nData Properties:")
print(f"  Contains negatives: {(X_train < 0).any()}")
print(f"  Contains zeros: {(X_train == 0).sum()}")
print(f"  Sparsity: {(X_train == 0).sum() / X_train.size * 100:.2f}%")

# Check if it could be image data
n_features = X_train.shape[1]
sqrt_feat = int(np.sqrt(n_features))
if sqrt_feat * sqrt_feat == n_features:
    print(f"\nPossible Format: {sqrt_feat}x{sqrt_feat} grayscale image")
    if n_features == 784:
        print(f"  → Likely MNIST-like dataset (28x28)")

# Visualize a few samples if it looks like image data
if sqrt_feat * sqrt_feat == n_features and n_features <= 1024:
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        img = X_train[i].reshape(sqrt_feat, sqrt_feat)
        ax.imshow(img, cmap='gray')
        ax.set_title(f'Label: {y_train[i]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Data Preprocessing

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

if X_unsup is not None:
    X_unsup_scaled = scaler.transform(X_unsup)

print("Data normalized using StandardScaler")
print(f"  Train - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")

## 4. Vision Transformer Model Architecture

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert 1D data into patches and embed them"""
    
    def __init__(self, input_dim, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = input_dim // patch_size
        
        # Linear projection
        self.projection = nn.Linear(patch_size, embed_dim)
        
        # Position embeddings
        self.position_embeddings = nn.Parameter(
            torch.randn(1, self.num_patches + 1, embed_dim) * 0.02
        )
        
        # Class token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        
    def forward(self, x):
        batch_size = x.shape[0]
        
        # Reshape into patches: (batch, input_dim) -> (batch, num_patches, patch_size)
        patches = x.view(batch_size, self.num_patches, self.patch_size)
        
        # Project to embedding dimension
        x = self.projection(patches)
        
        # Add class token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Add position embeddings
        x = x + self.position_embeddings
        
        return x

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention mechanism"""
    
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # Q, K, V projections
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape
        
        # Compute Q, K, V
        qkv = self.qkv(x).reshape(batch_size, num_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(batch_size, num_tokens, embed_dim)
        x = self.proj(x)
        x = self.dropout(x)
        
        return x

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer encoder block"""
    
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )
        
    def forward(self, x):
        # Self-attention with residual connection
        x = x + self.attn(self.norm1(x))
        # MLP with residual connection
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class VisionTransformer(nn.Module):
    """Vision Transformer for classification"""
    
    def __init__(self, input_dim, num_classes, patch_size=16, embed_dim=256, 
                 depth=6, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        
        self.input_dim = input_dim
        self.num_classes = num_classes
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(input_dim, patch_size, embed_dim)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        # Layer normalization
        self.norm = nn.LayerNorm(embed_dim)
        
        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x):
        # Patch embedding
        x = self.patch_embed(x)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Layer norm
        x = self.norm(x)
        
        # Extract class token
        cls_token_final = x[:, 0]
        
        # Classification
        logits = self.head(cls_token_final)
        
        return logits

print("Vision Transformer architecture defined!")

## 5. Masked Autoencoder for Pre-training (Optional)

In [ ]:
class MaskedAutoencoder(nn.Module):
    """Masked Autoencoder for self-supervised pre-training"""
    
    def __init__(self, input_dim, patch_size=16, embed_dim=256, 
                 depth=6, num_heads=8, mask_ratio=0.75):
        super().__init__()
        
        self.input_dim = input_dim
        self.patch_size = patch_size
        self.num_patches = input_dim // patch_size
        self.mask_ratio = mask_ratio
        
        # Encoder
        self.patch_embed = PatchEmbedding(input_dim, patch_size, embed_dim)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, 4.0, 0.0)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Decoder
        decoder_dim = embed_dim // 2
        self.decoder_embed = nn.Linear(embed_dim, decoder_dim)
        self.decoder_blocks = nn.ModuleList([
            TransformerBlock(decoder_dim, num_heads // 2, 4.0, 0.0)
            for _ in range(2)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_dim)
        self.decoder_pred = nn.Linear(decoder_dim, patch_size)
        
        # Mask token
        self.mask_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        
    def random_masking(self, x, mask_ratio):
        batch_size, num_patches_plus1, embed_dim = x.shape
        num_patches = num_patches_plus1 - 1
        
        num_keep = int(num_patches * (1 - mask_ratio))
        
        # Random shuffle
        noise = torch.rand(batch_size, num_patches, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        
        # Keep subset
        ids_keep = ids_shuffle[:, :num_keep]
        
        cls_tokens = x[:, :1, :]
        x_patches = x[:, 1:, :]
        
        x_unmasked = torch.gather(
            x_patches, dim=1, 
            index=ids_keep.unsqueeze(-1).expand(-1, -1, embed_dim)
        )
        
        x_unmasked = torch.cat([cls_tokens, x_unmasked], dim=1)
        
        # Create mask: 0 is keep, 1 is remove
        mask = torch.ones([batch_size, num_patches], device=x.device)
        mask[:, :num_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        
        return x_unmasked, mask, ids_restore
    
    def forward(self, x):
        # Embed
        x = self.patch_embed(x)
        
        # Mask
        x, mask, ids_restore = self.random_masking(x, self.mask_ratio)
        
        # Encode
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        
        # Decode
        x = self.decoder_embed(x)
        
        # Add mask tokens
        mask_tokens = self.mask_token.expand(x.shape[0], self.num_patches + 1 - x.shape[1], -1)
        mask_tokens = self.decoder_embed(mask_tokens)
        
        cls_token = x[:, :1, :]
        x_patches = x[:, 1:, :]
        x_full = torch.cat([x_patches, mask_tokens], dim=1)
        
        # Unshuffle
        x_full = torch.gather(
            x_full, dim=1,
            index=ids_restore.unsqueeze(-1).expand(-1, -1, x_full.shape[2])
        )
        
        x_full = torch.cat([cls_token, x_full], dim=1)
        
        # Decode
        for block in self.decoder_blocks:
            x_full = block(x_full)
        x_full = self.decoder_norm(x_full)
        
        # Predict
        pred = self.decoder_pred(x_full[:, 1:, :])
        
        return pred, mask

print("Masked Autoencoder for pre-training defined!")

## 6. Configuration

In [ ]:
# Get data properties
input_dim = X_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

# Find optimal patch size (must divide input_dim evenly)
optimal_patch_size = 16
for ps in range(16, 0, -1):
    if input_dim % ps == 0:
        optimal_patch_size = ps
        break

print(f"Input dimension: {input_dim}")
print(f"Number of classes: {num_classes}")
print(f"Optimal patch size: {optimal_patch_size}")
print(f"Number of patches: {input_dim // optimal_patch_size}")

# Hyperparameters
CONFIG = {
    # Model
    'patch_size': optimal_patch_size,
    'embed_dim': 256,
    'depth': 6,
    'num_heads': 8,
    'mlp_ratio': 4.0,
    'dropout': 0.1,
    
    # Pre-training (MAE)
    'pretrain_epochs': 30,
    'pretrain_lr': 1e-3,
    'pretrain_batch_size': 256,
    
    # Training
    'epochs': 100,
    'batch_size': 128,
    'lr': 5e-4,
    'weight_decay': 0.01,
    'val_split': 0.1,
    
    # Other
    'use_pretrain': USE_PRETRAIN,
}

print("\nConfiguration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 7. Pre-training with Masked Autoencoder (if unsupervised data available)

In [ ]:
if CONFIG['use_pretrain']:
    print("="*70)
    print("PRE-TRAINING WITH MASKED AUTOENCODER")
    print("="*70)
    
    # Create MAE model
    mae = MaskedAutoencoder(
        input_dim=input_dim,
        patch_size=CONFIG['patch_size'],
        embed_dim=CONFIG['embed_dim'],
        depth=CONFIG['depth'],
        num_heads=CONFIG['num_heads']
    ).to(device)
    
    # Create dataloader
    unsup_dataset = TensorDataset(torch.FloatTensor(X_unsup_scaled))
    unsup_loader = DataLoader(
        unsup_dataset,
        batch_size=CONFIG['pretrain_batch_size'],
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    # Optimizer
    optimizer = optim.AdamW(mae.parameters(), lr=CONFIG['pretrain_lr'], weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, CONFIG['pretrain_epochs'])
    
    # Training loop
    mae.train()
    pretrain_losses = []
    
    for epoch in range(CONFIG['pretrain_epochs']):
        epoch_loss = 0
        num_batches = 0
        
        for batch in tqdm(unsup_loader, desc=f'Epoch {epoch+1}/{CONFIG["pretrain_epochs"]}'):
            x = batch[0].to(device)
            
            # Forward
            pred, mask = mae(x)
            
            # Compute loss on masked patches
            batch_size = x.shape[0]
            target_patches = x.view(batch_size, mae.num_patches, mae.patch_size)
            
            loss = (pred - target_patches) ** 2
            loss = (loss * mask.unsqueeze(-1)).sum() / mask.sum()
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / num_batches
        pretrain_losses.append(avg_loss)
        
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.6f}, LR = {scheduler.get_last_lr()[0]:.6f}")
    
    # Plot pre-training loss
    plt.figure(figsize=(10, 5))
    plt.plot(pretrain_losses)
    plt.title('MAE Pre-training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()
    
    # Save pre-trained weights
    torch.save(mae.state_dict(), 'mae_pretrained.pth')
    print("\nPre-training completed! Weights saved.\n")
else:
    print("Skipping pre-training (no unsupervised data)\n")

## 8. Train Vision Transformer Classifier

In [ ]:
# Split into train and validation
num_train = len(X_train_scaled)
num_val = int(num_train * CONFIG['val_split'])

# Stratified split to maintain class distribution
from sklearn.model_selection import train_test_split
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_scaled, y_train, 
    test_size=CONFIG['val_split'], 
    stratify=y_train,
    random_state=SEED
)

print(f"Train: {len(X_train_split)} samples")
print(f"Validation: {len(X_val)} samples")

In [ ]:
# Create dataloaders
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_split),
    torch.LongTensor(y_train_split)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val),
    torch.LongTensor(y_val)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# Create Vision Transformer model
vit = VisionTransformer(
    input_dim=input_dim,
    num_classes=num_classes,
    patch_size=CONFIG['patch_size'],
    embed_dim=CONFIG['embed_dim'],
    depth=CONFIG['depth'],
    num_heads=CONFIG['num_heads'],
    mlp_ratio=CONFIG['mlp_ratio'],
    dropout=CONFIG['dropout']
).to(device)

# Load pre-trained weights if available
if CONFIG['use_pretrain']:
    try:
        pretrained_dict = torch.load('mae_pretrained.pth')
        model_dict = vit.state_dict()
        
        # Filter compatible weights
        pretrained_dict = {k: v for k, v in pretrained_dict.items() 
                          if k in model_dict and v.shape == model_dict[k].shape}
        
        model_dict.update(pretrained_dict)
        vit.load_state_dict(model_dict)
        print("Loaded pre-trained weights from MAE!\n")
    except:
        print("Could not load pre-trained weights. Training from scratch.\n")

# Count parameters
total_params = sum(p.numel() for p in vit.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(vit.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, CONFIG['epochs'])

# Training loop
print("="*70)
print("TRAINING VISION TRANSFORMER CLASSIFIER")
print("="*70 + "\n")

best_val_acc = 0.0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(CONFIG['epochs']):
    # Training
    vit.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for x, y in tqdm(train_loader, desc=f'Train Epoch {epoch+1}/{CONFIG["epochs"]}'):
        x, y = x.to(device), y.to(device)
        
        # Forward
        logits = vit(x)
        loss = criterion(logits, y)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        train_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        train_correct += (predicted == y).sum().item()
        train_total += y.size(0)
    
    train_acc = 100.0 * train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    vit.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = vit(x)
            loss = criterion(logits, y)
            
            val_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            val_correct += (predicted == y).sum().item()
            val_total += y.size(0)
    
    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler.step()
    
    # Record history
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1:3d}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, "
          f"Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, LR={scheduler.get_last_lr()[0]:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(vit.state_dict(), 'best_vit_model.pth')
        print(f"  ✓ New best model saved! (Val Acc: {val_acc:.2f}%)")

print(f"\nTraining completed! Best Val Acc: {best_val_acc:.2f}%")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(train_accs, label='Train Acc', linewidth=2)
axes[1].plot(val_accs, label='Val Acc', linewidth=2)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Generate Predictions on Test Set

In [ ]:
# Load best model
vit.load_state_dict(torch.load('best_vit_model.pth'))
vit.eval()

print("="*70)
print("GENERATING PREDICTIONS ON TEST SET")
print("="*70)

# Create test dataloader
test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled))
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2
)

# Predict
predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Predicting'):
        x = batch[0].to(device)
        logits = vit(x)
        _, predicted = torch.max(logits, 1)
        predictions.extend(predicted.cpu().numpy())

print(f"\nGenerated {len(predictions)} predictions")

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'Id': range(len(predictions)),
    'Prediction': predictions
})

submission.to_csv('submission.csv', index=False)

print("Submission file created: submission.csv")
print("\nFirst 10 predictions:")
print(submission.head(10))

## 10. Summary and Results

In [ ]:
print("="*70)
print("SUMMARY")
print("="*70)
print(f"\nModel: Vision Transformer")
print(f"  Parameters: {total_params:,}")
print(f"  Patch size: {CONFIG['patch_size']}")
print(f"  Embedding dim: {CONFIG['embed_dim']}")
print(f"  Depth: {CONFIG['depth']}")
print(f"  Attention heads: {CONFIG['num_heads']}")

print(f"\nTraining:")
print(f"  Best validation accuracy: {best_val_acc:.2f}%")
print(f"  Pre-training used: {CONFIG['use_pretrain']}")
print(f"  Total epochs: {CONFIG['epochs']}")

print(f"\nPredictions:")
print(f"  Test samples: {len(predictions)}")
print(f"  Prediction distribution:")
for class_id, count in enumerate(np.bincount(predictions)):
    print(f"    Class {class_id}: {count} ({count/len(predictions)*100:.1f}%)")

print("\n" + "="*70)
print("Ready to submit to Kaggle!")
print("="*70)